**Notebook version: 1** — Claude will state the version number after editing any cell in this notebook. If the number here doesn't match what Claude just said, your editor has a stale copy: close this tab and reopen it (don't rely on "close without saving" — autosave can already have overwritten the file with the stale version before you close) before running anything.

# fiftyone_near_dup_inspection.ipynb — post-DEC-089 near-duplicate threshold review

**When to use this:** after a real `dedup.py --full-scale` run (DEC-089) has produced `dataset/reports/dedup_report.json`, to visually judge whether `near_duplicate_threshold: 0.2` is actually right for the `mobilenet-v2-imagenet-torch` embedding space used — that guidance is FiftyOne Brain's own default, explicitly left unverified for this embedding model (see the report's own `near_duplicate_threshold_source` field).

**What it does:** loads near-duplicate groups from the report into a throwaway FiftyOne dataset (kept + duplicate images, tagged with `group`/`role`/`distance`/`source`), lets you mark false positives (individually or in bulk by source), and exports what you've marked to `dataset/reports/near_duplicate_false_positives.json`.

**Not for:** the main box/label review pass (`labels_reviewed/` promotion, exclude/accept tagging on ground truth) — use `fiftyone_review_processed.ipynb` for that. This notebook is read-only against `dataset/merged/` and `dataset/processed/`; it only ever writes `near_duplicate_false_positives.json`.

**Split out from `fiftyone_review_processed.ipynb` on 2026-08-24** to keep the two concerns separate — that notebook already covers a lot of ground (box review, mistakenness, write-back), and this is a distinct, self-contained tool with its own throwaway datasets.

In [1]:
# Imports
import json
import random
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    """Walk up from `start` to find the repo root (has config/ + AGENTS.md).

    Needed because notebooks live in notebooks/, not the repo root, and
    Jupyter's working directory depends on how it was launched -- this
    makes the scripts.* import below robust regardless of that.
    """
    for parent in [start, *start.parents]:
        if (parent / "config").is_dir() and (parent / "AGENTS.md").is_file():
            return parent
    raise RuntimeError("Could not locate repo root from notebook cwd.")


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

import fiftyone as fo
from fiftyone import ViewField as F

from scripts.utils.file_utils import merged_dir, reports_dir

/opt/anaconda3/envs/second-vision/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Real numbers so far (2026-08-22/24, DEC-089)

**Report-level (`dataset/reports/dedup_report.json`):** `images_checked`/`near_duplicate_sample_size` both 66,907 (true full-scale, no sampling). Exact-duplicates: 22 groups / 23 files. Near-duplicates (threshold=0.2, unverified for this embedding space): **5,868 groups / 12,692 files** (~19% of the pool).

**Per-class impact if every flagged duplicate were excluded** (hypothetical — not what `split.py` does today, it only keeps groups together across splits):

| Class | Current | Flagged | If deduped | % dropped |
|---|---:|---:|---:|---:|
| Potholes | 2,661 | 1,834 | **827** | **68.9%** |
| Pedestrian Lane | 2,745 | 1,383 | **1,362** | **50.4%** |
| Doors | 5,819 | 2,976 | 2,843 | 51.1% |
| Elevator | 3,191 | 1,419 | 1,772 | 44.5% |
| Stairs | 2,371 | 772 | 1,599 | 32.6% |
| Trash Bins | 1,683 | 386 | 1,297 | 22.9% |
| Vehicle | 10,000 | 1,281 | 8,719 | 12.8% |
| Person | 10,000 | 1,118 | 8,882 | 11.2% |
| Motorcycle | 8,281 | 905 | 7,376 | 10.9% |
| Tricycle | 3,798 | 304 | 3,494 | 8.0% |
| Animals | 7,319 | 516 | 6,803 | 7.1% |
| Shelf | 2,386 | 117 | 2,269 | 4.9% |
| Chairs | 2,984 | 139 | 2,845 | 4.7% |
| Pole | 2,186 | 91 | 2,095 | 4.2% |
| Tables | 5,355 | 178 | 5,177 | 3.3% |
| Bicycle | 3,655 | 116 | 3,539 | 3.2% |

**Potholes and Pedestrian Lane would fall below the project's 1,500-image floor** if every flagged duplicate were excluded blindly — real signal that any future exclusion step needs per-class judgment, not a blanket rule.

**Potholes by source** — the 68.9% class-level drop is *not* evenly spread:

| Source | Selected | Flagged | Remaining | % dropped |
|---|---:|---:|---:|---:|
| `roboflow_pothole_voxrl` | 665 | 630 | 35 | **94.7%** |
| `dataset_ninja_road_damage_detector` | 1,331 | 977 | 354 | 73.4% |
| `dataset_ninja_pothole_detection` | 665 | 227 | 438 | 34.1% |

`roboflow_pothole_voxrl` losing 94.7% looks less like genuine duplication and more like an embedding-space collision — pothole photos are visually narrow (asphalt, cracks, similar framing), so different potholes may cluster as "near-duplicates" simply for looking like the same *category*, not the same photo.

**Bulk-cleared as likely false positives:** `exdark` (728 flagged) + `open_images` (840 flagged) = **1,568 duplicate-role images** tagged `false_positive`, on the reasoning that both are established, well-curated, high-diversity sources unlikely to be genuinely this redundant. Written to `dataset/reports/near_duplicate_false_positives.json`.

**Important interpretive caveat (confirmed directly against real data, not hypothetical):** `near_duplicates[i]["duplicates"][j]["distance"]` is **not reliable** as "how close this match was." FiftyOne Brain's `neighbors_map` reports distance to the nearest *surviving unique* neighbor (a post-hoc query), not necessarily the neighbor that actually triggered the original sub-0.2 flag — confirmed concretely when the bulk-tagged exdark/open_images export came back with a distance range of **1.64 to 8.10**, far past the 0.2 cutoff that must have originally flagged them. The `kept`/`duplicate` group *membership* is still accurate; only the displayed `distance` number can be misleading. Don't use exported distance values to infer where the "real" threshold boundary is.

# Near-duplicate inspection (post-DEC-089)

**Why this is here:** the real, full-scale `dedup.py` run (DEC-089, 2026-08-22) flagged 5,868 near-duplicate groups (12,692 files, ~19% of the 66,907-image pool) at `near_duplicate_threshold: 0.2` — FiftyOne Brain's own default, left **unverified** for the `mobilenet-v2-imagenet-torch` embedding space actually used (the report's own `near_duplicate_threshold_source` field flags this explicitly). This is a local, GPU-free visual check of whether 0.2 looks right — not a re-run of dedup itself.

**What it loads:** every near-duplicate group from the report (all ~18,500 involved images), each tagged with its `group` id, `role` (`kept` vs `duplicate`), `distance` (for duplicate members), `group_max_distance` (same value across every member of a group), and `source`. The view is sorted by `group_max_distance` descending. **Note the interpretive caveat in the numbers cell above** before reading too much into `distance`/`group_max_distance` values specifically — group membership is reliable, the distance numbers are not always.

**Caution** (documented pain point in this project, not hypothetical): make sure no *other* Jupyter kernel is currently connected to the FiftyOne App before running this — a forgotten background kernel holding a stale dataset has silently hijacked the shared App view before (see DECISIONS.md's App-reverts-to-hovyc saga). This cell's own `session.view = spotcheck_view` line guards against the same class of bug the rest of this notebook already guards against.

In [2]:
# Loads EVERY near-duplicate group from dedup_report.json for full visual
# inspection -- see the markdown cell above for why. Read-only: does not
# touch dataset/processed/, dataset/merged/, or any report file.

with open(reports_dir() / "dedup_report.json") as f:
    dedup_report = json.load(f)

near_groups = dedup_report["near_duplicates"]

SPOTCHECK_DATASET_NAME = "near_dup_full_inspection"
if SPOTCHECK_DATASET_NAME in fo.list_datasets():
    fo.delete_dataset(SPOTCHECK_DATASET_NAME)
spotcheck_dataset = fo.Dataset(SPOTCHECK_DATASET_NAME)

images_dir = merged_dir() / "images"
samples = []
missing = 0
for gi, group in enumerate(near_groups):
    group_max_distance = max(d["distance"] for d in group["duplicates"])
    group_id = f"group_{gi:05d}"

    kept_path = images_dir / group["kept"]
    if kept_path.exists():
        s = fo.Sample(filepath=str(kept_path))
        s["group"] = group_id
        s["role"] = "kept"
        s["group_max_distance"] = group_max_distance
        s["source"] = group["kept"].split("__", 1)[0]
        samples.append(s)
    else:
        missing += 1

    for dup in group["duplicates"]:
        dup_path = images_dir / dup["filename"]
        if dup_path.exists():
            s = fo.Sample(filepath=str(dup_path))
            s["group"] = group_id
            s["role"] = "duplicate"
            s["distance"] = dup["distance"]
            s["group_max_distance"] = group_max_distance
            s["source"] = dup["filename"].split("__", 1)[0]
            samples.append(s)
        else:
            missing += 1

spotcheck_dataset.add_samples(samples)
print(f"{len(near_groups)} groups, {len(samples)} images loaded"
      + (f" ({missing} missing on disk, skipped)" if missing else "") + ".")
print("Filter/group by the `source` field in the App sidebar to focus on one "
      "dataset at a time (e.g. source == \"roboflow_pothole_voxrl\").")

spotcheck_view = spotcheck_dataset.sort_by("group_max_distance", reverse=True)
spotcheck_session = fo.launch_app(spotcheck_view, auto=False)
spotcheck_session.view = spotcheck_view
print(f"App session bound to {SPOTCHECK_DATASET_NAME!r} view -- most borderline groups first.")

 100% |█████████████| 18560/18560 [1.1s elapsed, 0s remaining, 16.5K samples/s]         
5868 groups, 18560 images loaded.
Filter/group by the `source` field in the App sidebar to focus on one dataset at a time (e.g. source == "roboflow_pothole_voxrl").
Session launched. Run `session.show()` to open the App in a cell output.
App session bound to 'near_dup_full_inspection' view -- most borderline groups first.


In [3]:
spotcheck_session

Dataset:          near_dup_full_inspection
Media type:       image
Num samples:      18560
Selected samples: 0
Selected labels:  0
Session URL:      http://localhost:5151/
View stages:
    1. SortBy(field_or_expr='group_max_distance', reverse=True, create_index=False)

### Bulk-mark a whole source as false positive (optional)

For sources you're confident are well-curated/distinct enough that being flagged as a near-duplicate is itself suspicious (e.g. exdark, open_images — established, high-quality datasets, unlikely to be genuinely this redundant) rather than reviewing group-by-group. Only tags `duplicate`-role samples (the ones actually flagged as duplicating something) — does not touch `kept` reference images, and doesn't catch cases where a bulk-cleared source's image is the `kept` side of a group with some *other* source flagged against it (check the per-source counts above/filter by `source` in the App if you want to look at those separately).

Edit `BULK_FALSE_POSITIVE_SOURCES` below, then run. Safe to re-run with a different list any time — it only adds the `false_positive` tag, never removes one, and the write-back cell further down always reflects current tags.

In [ ]:
from fiftyone import ViewField as F

# Sources judged well-curated/distinct enough that a duplicate flag against
# them is itself suspicious. Edit freely.
BULK_FALSE_POSITIVE_SOURCES = ["exdark", "open_images", "roboflow_cv_project_hovyc", "crowdhuman", "dataset_ninja_pothole_detection", "roboflow_pothole_voxrl", "dataset_ninja_road_damage", "roboflow_trashcan_detection_pihfn"]

bulk_view = spotcheck_dataset.match(
    (F("role") == "duplicate") & F("source").is_in(BULK_FALSE_POSITIVE_SOURCES)
)
bulk_view.tag_samples("false_positive")
print(f"Tagged {len(bulk_view)} duplicate-role samples from {BULK_FALSE_POSITIVE_SOURCES} as false_positive.")

Tagged 2472 duplicate-role samples from ['exdark', 'open_images', 'roboflow_cv_project_hovyc', 'crowdhuman', 'dataset_ninja_pothole_detection', 'roboflow_pothole_voxrl', 'dataset_ninja_road_damage'] as false_positive.


### Mark false positives, then write them back

While browsing above: select any image you judge as **not actually a near-duplicate** and tag it `false_positive` (App's tag icon at the bottom of the grid, or press `T`) — same native FiftyOne tagging this notebook already uses for `exclude`/`accept` elsewhere (DEC-078/079), nothing new to learn.

Run the cell below whenever you want to export what you've tagged so far — it reads every `duplicate`-role sample tagged `false_positive`, and writes the (kept, duplicate, distance) triples to `dataset/reports/near_duplicate_false_positives.json`, plus prints the distance range they span. **Per the interpretive caveat above, that distance range is not a reliable signal for where the threshold should sit** — treat the exported list itself (which pairs got marked) as the useful output, not the distance numbers attached to it. Safe to re-run any time — it always reflects current tags, doesn't touch `dedup_report.json` or anything else.

In [ ]:
# Exports every `false_positive`-tagged duplicate as a (kept, duplicate,
# distance) triple. Read-only against dedup_report.json; writes only to
# near_duplicate_false_positives.json -- does not touch dataset/merged/,
# hide_duplicates, or split.py's duplicate grouping. Deciding whether/how
# those specific pairs should stop being treated as duplicates downstream
# is a separate follow-up, not made here.
from pathlib import Path

false_positive_view = spotcheck_dataset.match_tags("false_positive")

reviewed = []
for sample in false_positive_view:
    if sample["role"] != "duplicate":
        continue  # only duplicate-role samples carry a kept/distance pair to record
    group_idx = int(sample["group"].split("_")[1])
    group = near_groups[group_idx]
    reviewed.append({
        "kept": group["kept"],
        "duplicate": Path(sample.filepath).name,
        "distance": sample["distance"],
        "group": sample["group"],
    })

fp_output_path = reports_dir() / "near_duplicate_false_positives.json"
with open(fp_output_path, "w") as f:
    json.dump(reviewed, f, indent=2)

print(f"{len(reviewed)} pairs tagged false_positive, written to {fp_output_path}")
if reviewed:
    distances = [r["distance"] for r in reviewed]
    print(f"Distance range of false positives: {min(distances):.4f} - {max(distances):.4f}")

In [ ]:
fo.close_app()